# THÊM VLM vào CUỐI inference X-VLM (đẩy R@1 / mAP)

**Dán 3 cell dưới vào CUỐI notebook `aicity-inference-v1`** — ngay sau cell đã ghi `answer.txt`.
Vẫn là 1 luồng inference + evaluate như cũ, **chỉ thêm 1 bước VLM**: Qwen2-VL-2B nhìn **top-10 ảnh**
mỗi query rồi chọn ảnh đúng nhất → đưa lên **rank-1**.

Qwen chạy bằng `!python` (**tiến trình con**) nên transformers mới của nó **không đụng** X-VLM trong kernel.
Chỉ cần thêm 1 input: **Qwen2-VL-2B-Instruct** (Kaggle Models). 2B fp16 → 1×T4, **không quantize**, ~0.5-1h.

In [ ]:
# [VLM-1] Tim answer.txt + query files + GT + model Qwen (chay SAU khi pipeline da tao answer.txt)
import glob, os
def _one(p):
    h = sorted(glob.glob(f"/kaggle/working/**/{p}", recursive=True)) + \
        sorted(glob.glob(f"/kaggle/input/**/{p}", recursive=True))
    return h[0] if h else None
ANSWER = _one("answer.txt"); QJSON = _one("query_text.json"); QIDX = _one("query_index.txt"); GT = _one("ground_truth.txt")
qcfg = [p for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "qwen" in p.lower()]
QWEN = os.path.dirname(qcfg[0]) if qcfg else "Qwen/Qwen2-VL-2B-Instruct"
assert QJSON and QIDX and GT, f"Thieu input: QJSON={QJSON} QIDX={QIDX} GT={GT}"
print("QWEN   :", QWEN, "(local)" if qcfg else "(se tai HuggingFace)")
print("ANSWER :", ANSWER or "(chua co -> se tao sau khi pipeline chay; binh thuong)")

In [ ]:
# [VLM-2] Tai engine qwen_rerank.py tu GitHub (Internet ON)
import os
!wget -q -O /kaggle/working/qwen_rerank.py https://raw.githubusercontent.com/Khanhhh239/Model_XVLM_Training/main/scripts/qwen_rerank.py
print("qwen_rerank.py:", os.path.getsize("/kaggle/working/qwen_rerank.py"), "bytes")

In [ ]:
# [VLM-3] Cai transformers moi + chay VLM re-rank (subprocess -> khong dung X-VLM)
import os
assert ANSWER, "Chua thay answer.txt — phai chay het pipeline o tren TRUOC roi moi chay cell nay."
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
!pip -q install -U "transformers>=4.49.0" "accelerate>=0.34" qwen-vl-utils hf_transfer
cmd = (f'python /kaggle/working/qwen_rerank.py --answer "{ANSWER}" --query-json "{QJSON}" '
       f'--query-index "{QIDX}" --gt "{GT}" --image-root /kaggle/input --model "{QWEN}" '
       f'--out /kaggle/working/outputs/answer_vlm.txt --topk 10')
print(cmd)
!{cmd}
# Bang mAP/R@1/R@5/R@10 truoc-sau in ngay duoi; answer_vlm.txt = ban da day VLM len rank-1.

## Cách chạy (Save & Run All → đi ngủ)
1. **Settings**: GPU **T4** · Internet **ON** · Add Input thêm **Qwen2-VL-2B-Instruct** (Kaggle Models).
2. Dán 3 cell trên vào **cuối** notebook → **Save Version → Save & Run All (Commit)**.
3. Tắt máy/đi ngủ — Kaggle chạy trên server (~1-1.5h, trong giới hạn 9h).
4. Xong: mở version → tab **Output** → tải `answer_vlm.txt`; bảng `X-VLM vs +VLM` nằm trong **log**.

**Lưu ý:** nếu mỗi dòng `answer.txt` có query-id ở đầu → thêm `--skip-first-col` vào `cmd` ở [VLM-3].
Số trên no-distractor + caption chi tiết → cao hơn đề thi thật.